# 將所有JSON檔案壓縮成單一ZIP檔案

支援大量檔案的高效壓縮，並顯示進度

## 特點
- 將所有JSON檔案壓縮成單一ZIP檔案
- 支援進度顯示
- 自動驗證ZIP檔案完整性
- 顯示壓縮率統計

In [1]:
import zipfile
from pathlib import Path
from datetime import datetime, timezone, timedelta
import json

# Set timezone to UTC+8 (Asia/Taipei)
TZ_UTC8 = timezone(timedelta(hours=8))

print("ZIP壓縮工具已載入")

ZIP壓縮工具已載入


## 配置參數

In [2]:
# 輸入輸出路徑
INPUT_JSON_DIR = Path("data/json_files")  # 從步驟1的輸出目錄讀取
OUTPUT_DIR = Path("data/zipped")  # ZIP檔案輸出目錄
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now(TZ_UTC8).strftime('%Y%m%d_%H%M%S')
OUTPUT_ZIP = OUTPUT_DIR / f"batch_input_{timestamp}.zip"

# 壓縮參數
COMPRESSION_LEVEL = 6  # 0-9, 9為最高壓縮率但最慢

print(f"輸入目錄: {INPUT_JSON_DIR}")
print(f"輸出ZIP: {OUTPUT_ZIP}")
print(f"壓縮等級: {COMPRESSION_LEVEL}")

輸入目錄: data/json_files
輸出ZIP: data/zipped/batch_input_20260412_223409.zip
壓縮等級: 6


## 檢查輸入檔案

In [3]:
# 檢查目錄是否存在
if not INPUT_JSON_DIR.exists():
    raise FileNotFoundError(f"找不到輸入目錄: {INPUT_JSON_DIR}")

# 取得所有JSON檔案
print("掃描JSON檔案...")
json_files = sorted(INPUT_JSON_DIR.glob("*.json"))
total_files = len(json_files)

if total_files == 0:
    raise ValueError(f"在 {INPUT_JSON_DIR} 中找不到JSON檔案")

print(f"找到 {total_files:,} 個JSON檔案")

# 計算原始總大小
print("計算原始檔案大小...")
original_size = sum(f.stat().st_size for f in json_files)
print(f"原始總大小: {original_size / (1024**2):.2f} MB ({original_size / (1024**3):.2f} GB)")

掃描JSON檔案...
找到 1,000 個JSON檔案
計算原始檔案大小...
原始總大小: 31.32 MB (0.03 GB)


## 執行壓縮

In [4]:
print("\n開始壓縮...")
start_time = datetime.now(TZ_UTC8)

files_zipped = 0

# 建立ZIP檔案
with zipfile.ZipFile(
    OUTPUT_ZIP,
    'w',
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=COMPRESSION_LEVEL
) as zipf:
    
    # 壓縮所有JSON檔案
    for json_file in json_files:
        # 使用相對路徑作為ZIP內的檔案名稱
        arcname = json_file.name
        zipf.write(json_file, arcname=arcname)
        
        files_zipped += 1
        
        # 每1000個檔案顯示一次進度
        if files_zipped % 1000 == 0:
            print(f"已壓縮: {files_zipped:,} / {total_files:,} 檔案 ({files_zipped/total_files*100:.1f}%)")

end_time = datetime.now(TZ_UTC8)
duration = (end_time - start_time).total_seconds()

print(f"\n✅ 壓縮完成！")
print(f"已壓縮檔案數: {files_zipped:,}")
print(f"壓縮時間: {duration:.2f} 秒 ({duration/60:.2f} 分鐘)")
print(f"壓縮速度: {files_zipped / duration:.2f} 檔案/秒")


開始壓縮...
已壓縮: 1,000 / 1,000 檔案 (100.0%)

✅ 壓縮完成！
已壓縮檔案數: 1,000
壓縮時間: 2.08 秒 (0.03 分鐘)
壓縮速度: 480.42 檔案/秒


## 壓縮統計

In [5]:
# 取得壓縮後大小
compressed_size = OUTPUT_ZIP.stat().st_size
compression_ratio = (1 - compressed_size / original_size) * 100

print("=" * 60)
print("壓縮統計")
print("=" * 60)
print(f"總檔案數: {total_files:,}")
print(f"已壓縮檔案數: {files_zipped:,}")
print(f"原始大小: {original_size / (1024**2):.2f} MB ({original_size / (1024**3):.2f} GB)")
print(f"壓縮後大小: {compressed_size / (1024**2):.2f} MB ({compressed_size / (1024**3):.2f} GB)")
print(f"壓縮率: {compression_ratio:.2f}%")
print(f"壓縮時間: {duration:.2f} 秒 ({duration/60:.2f} 分鐘)")
print(f"壓縮速度: {files_zipped / duration:.2f} 檔案/秒")
print("=" * 60)

壓縮統計
總檔案數: 1,000
已壓縮檔案數: 1,000
原始大小: 31.32 MB (0.03 GB)
壓縮後大小: 14.62 MB (0.01 GB)
壓縮率: 53.31%
壓縮時間: 2.08 秒 (0.03 分鐘)
壓縮速度: 480.42 檔案/秒


## 驗證ZIP檔案

In [6]:
print("\n驗證ZIP檔案完整性...")

try:
    with zipfile.ZipFile(OUTPUT_ZIP, 'r') as zipf:
        # 測試ZIP檔案
        bad_file = zipf.testzip()
        
        if bad_file is not None:
            print(f"❌ ZIP檔案損壞: {bad_file}")
        else:
            print("✅ ZIP檔案完整性驗證通過")
        
        # 檢查檔案數量
        zip_file_count = len(zipf.namelist())
        print(f"\nZIP內檔案數: {zip_file_count:,}")
        
        if zip_file_count != total_files:
            print(f"⚠️ 警告: ZIP檔案數量不符 (預期 {total_files:,}, 實際 {zip_file_count:,})")
        else:
            print("✅ 檔案數量驗證通過")
        
        # 顯示前5個和後5個檔案名稱
        file_list = zipf.namelist()
        print(f"\n前5個檔案:")
        for name in file_list[:5]:
            print(f"  - {name}")
        
        print(f"\n後5個檔案:")
        for name in file_list[-5:]:
            print(f"  - {name}")
        
except Exception as e:
    print(f"❌ 驗證ZIP檔案時發生錯誤: {e}")
    raise


驗證ZIP檔案完整性...
✅ ZIP檔案完整性驗證通過

ZIP內檔案數: 1,000
✅ 檔案數量驗證通過

前5個檔案:
  - record_0000000.json
  - record_0000001.json
  - record_0000002.json
  - record_0000003.json
  - record_0000004.json

後5個檔案:
  - record_0000995.json
  - record_0000996.json
  - record_0000997.json
  - record_0000998.json
  - record_0000999.json


## 測試解壓縮（可選）

In [ ]:
# 測試讀取ZIP內的第一個JSON檔案
print("\n測試讀取ZIP內的JSON檔案...")

with zipfile.ZipFile(OUTPUT_ZIP, 'r') as zipf:
    # 讀取第一個JSON檔案
    first_file = zipf.namelist()[0]
    print(f"讀取檔案: {first_file}")
    
    with zipf.open(first_file) as f:
        json_data = json.load(f)
        print(f"\nJSON結構:")
        print(f"  input_data項目數: {len(json_data['input_data'])}")
        for item in json_data['input_data']:
            print(f"    - {item['id']}: {len(item['values'][0])} 個值")

print("\n✅ ZIP檔案可正常讀取")

## 儲存統計資訊

In [ ]:
# 儲存統計資訊
stats_file = Path("logs") / f"zipping_stats_{timestamp}.json"
stats_file.parent.mkdir(parents=True, exist_ok=True)

stats_data = {
    "total_files": total_files,
    "files_zipped": files_zipped,
    "original_size_bytes": original_size,
    "original_size_mb": original_size / (1024**2),
    "original_size_gb": original_size / (1024**3),
    "compressed_size_bytes": compressed_size,
    "compressed_size_mb": compressed_size / (1024**2),
    "compressed_size_gb": compressed_size / (1024**3),
    "compression_ratio_percent": compression_ratio,
    "compression_level": COMPRESSION_LEVEL,
    "duration_seconds": duration,
    "compression_speed_files_per_sec": files_zipped / duration,
    "input_dir": str(INPUT_JSON_DIR),
    "output_zip": str(OUTPUT_ZIP),
    "start_time": start_time.isoformat(),
    "end_time": end_time.isoformat(),
}

with open(stats_file, 'w') as f:
    json.dump(stats_data, f, indent=2, ensure_ascii=False)

print(f"\n統計資訊已儲存至: {stats_file}")

## 總結

In [ ]:
print("\n" + "=" * 60)
print("壓縮完成總結")
print("=" * 60)
print(f"✅ 已將 {files_zipped:,} 個JSON檔案壓縮成單一ZIP檔案")
print(f"✅ ZIP檔案: {OUTPUT_ZIP}")
print(f"✅ 檔案大小: {compressed_size / (1024**3):.2f} GB")
print(f"✅ 壓縮率: {compression_ratio:.2f}%")
print("=" * 60)
print(f"\n下一步: 使用 4_upload_and_create_job.ipynb 上傳到watsonx.ai並建立批次作業")